`r format(Sys.time(), '🗓️ %d %B, %Y 🕔 %H:%M')`

In [ ]:
library(DT)
library(magrittr)
library(tidyr)
library(scales)

In [ ]:
infile <- params$statsfile
dataL <- readLines(infile)
bcf <- gsub(".stats", ".bcf", basename(infile))
basedir <- dirname(normalizePath(infile))
plotdir <- paste0(normalizePath(params$plotdir), "/")

# Samples
For convenience, this report consolidates the various outputs from `STITCH` and `bcftools` into a single document.
This reflects the general information stored in the records of `r bcf`. Since STITCH
requires only bi-allelic SNPs as input, you should not expect to see any other types
of variants (MNPs, indels, _etc._). For the wider images, you may right-click them and open
them in a new tab/window to view them at their original sizes, which may prove to be more useful in some cases.

Extra parameters provided: `r params$extra`

In [ ]:
.snL <- grepl("^SN", dataL)
sn <- read.table(text=dataL[.snL], sep = "\t")[, 3:4]
names(sn) <- c("Metric", "Number")
sn$Metric <- gsub("number of ", "", sn$Metric)
sn$Metric <- gsub(":", "", sn$Metric)
rownames(sn) <- sn$Metric
sn <- as.data.frame(t(sn[2]))
rownames(sn) <- NULL

## model stats

In [ ]:
list(
  color = "light",
  value = params$model
)

In [ ]:
list(
  color = "info",
  value = params$usebx
)

In [ ]:
list(
  color = "info",
  value = params$k
)

In [ ]:
list(
  color = "info",
  value = params$s
)

In [ ]:
list(
  color = "info",
  value = params$ngen
)

In [ ]:
list(
  color = "info",
  value = prettyNum(params$bxlimit, big.mark = ",")
)

## result stats

In [ ]:
list(
  color = "light",
  value = prettyNum(sn$samples[1], big.mark = ",")
)

In [ ]:
list(
  color = "#dfdfdf",
  value = prettyNum(sn$records[1], big.mark = ",")
)

In [ ]:
list(
  color = "#dfdfdf",
  value = prettyNum(sn[3,1], big.mark = ",")
)

In [ ]:
list(
  color = "#dfdfdf",
  value = prettyNum(sn$SNPs[1], big.mark = ",")
)

In [ ]:
list(
  color = "#dfdfdf",
  value = prettyNum(sn$MNPs[1], big.mark = ",")
)

## Per Sample Stats
::: {.card title="Per-Sample Statistics" expandable="true"}

This table contains variant statistics per individual, corresponding 
to the `r sn$samples[1]` samples in `r bcf`. 

In [ ]:
.pscL <- grepl("^PSC", dataL)
psc <- read.table(text=dataL[.pscL])[ ,3:14]
names(psc) <- c("Sample", "HomozygousRef", "HomozygousAlt", "Heterozygotes", "Transitions", "Transversions", "Indels",	"MeanDepth", "Singletons",	"HapRef", "HapAlt", "Missing")
psc$Homozygotes <- psc$HomozygousRef + psc$HomozygousAlt
DT::datatable(
  psc[,c(1,12,2,3,13,4,5,6,9)],
  rownames = F, 
  filter = "top", 
  extensions = 'Buttons', 
  fillContainer = T,
  options = list(
    dom = 'Brtp',
    buttons = list(list(extend = "csv",filename = "impute_per_sample")),  
    scrollX = TRUE
  )
)

:::

# QC
##
TOP
: Gives -log10 (HWE p-value)

MIDDLE
: Shows `INFO` score, which is a measure of the confidence the imputation has in its performance

BOTTOM
: Shows allele frequency, which can be useful to get a 
sense of the LD structure with the distribution of allele frequencies,
as well as its relationship to imputation performance. Can also be used
to test how various parameter choices affect imputation. For example,
for species that have recently been through bottlenecks, allele frequencies
are often highly correlated locally, visible as multiple nearby SNPs that 
have the same allele frequency. Choices of imputation parameters that increase
the tightness in the spread of these allele frequencies often correlate with
increased imputation performance.

_- adapted from the STITCH [documentation](https://github.com/rwdavies/STITCH/blob/5a5ba98442a105548587ed8cafbf00aece212c94/plots.md)_.

###

In [ ]:
QC <- list.files(path = plotdir, pattern = "QC", full.names = TRUE)
knitr::include_graphics(QC[2])

## Imputation QC Plot
Similar to the previous plots, but plotting each of the three against each
other. Useful for getting an overall sense of the distribution of the
metrics, particularly `INFO`, which can be useful towards thinking about a
threshold for filtering out variants after imputation. For example, in the
middle plot with estimated allele frequency vs info, if you expect your data
to cluster into a small range of allele frequencies, and your data seems
concentrated in those frequencies for high `INFO`, you can think about an
`INFO` score cutoff that allows you to capture most of these variants.

_- excerpt from the STITCH [documentation](https://github.com/rwdavies/STITCH/blob/5a5ba98442a105548587ed8cafbf00aece212c94/plots.md)_.

###

In [ ]:
knitr::include_graphics(QC[1])

# Real vs Estimated
##
Scatter plot of real allele frequency as estimated from the pileup of
sequencing reads (x-axis) and estimated allele frequency from the
posterior of the model (y-axis). This is per-SNP, the sum of the average
usage of each ancestral haplotype times the probability that ancestral
haplotype emits an alternate base, divided by the number of samples. One
should generally see good agreement between these. Note that these are at
"good SNPs" with `INFO` score > `0.4` and HWE p-value > `1x10^-6` (the 
later criterion in particular might not be appropriate for all settings).

_- excerpt from the STITCH [documentation](https://github.com/rwdavies/STITCH/blob/5a5ba98442a105548587ed8cafbf00aece212c94/plots.md)_.

###

In [ ]:
goodonly <- list.files(path = plotdir, pattern = "goodonly", full.names = TRUE)
knitr::include_graphics(goodonly[1])

# Expected Haplotypes
##
::: {.card title="Expected Haplotypes" expandable="true"}
Across the region being imputed (x-axis, physical position), either the
fraction (normal version) or log10 of the expected number of haplotypes
each sample carries from each ancestral haplotype. The most important
thing to see in this plot is that there is not drastic, but rather gradual,
movement between the relative proportions of the ancestral haplotypes.
Large swings in ancestral haplotype sums indicate the heuristics are working
sub-optimally (though this could also be interesting biology!). It is also
useful to check here how many ancestral haplotypes are being used, to see if
some could be trimmed, for example if some are very infrequently used, though
this is rare, as STITCH will try to fill them up. 

_- excerpt from the STITCH [documentation](https://github.com/rwdavies/STITCH/blob/5a5ba98442a105548587ed8cafbf00aece212c94/plots.md)_.

In [ ]:
hapsum <- list.files(path = plotdir, pattern = "hapSum", full.names = TRUE)
knitr::include_graphics(hapsum)

:::

##
:::{.card title="Alpha Matrix" expandable="true"}
Likely not informative for general use. `alphaMat` is the name of the
internal `STITCH` variable that stores the probability of jumping into
an ancestral haplotype conditional on a jump. Un-normalized is with
respect to movement of all samples, while normalized has sum 1. 

_- excerpt from the STITCH [documentation](https://github.com/rwdavies/STITCH/blob/5a5ba98442a105548587ed8cafbf00aece212c94/plots.md)_.

In [ ]:
alphamat <- list.files(path = plotdir, pattern = "alphaMat", full.names = TRUE)
knitr::include_graphics(alphamat)

:::